# 04 — 非线性隐变量：直接数值积分
唯一结构性修改：将 $wx+b$ 换成神经网络 $f_\theta(x)$。先验和观测噪声不变。联合密度依旧容易逐点计算，但边缘密度与后验一般无解析解。由于隐变量只有一维，仍可用网格近似积分，并直接最大化观测似然。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from utils import generate_observations, compare, plot_history

# 共享数据，只观测 y；不读取真实生成类别。
rng = np.random.default_rng(42)
y = generate_observations(rng)

## 1. 定义模型与计算边缘密度
沿用第 03 章的标准正态先验与固定二维观测噪声，仅将均值函数替换为神经网络。无法再直接使用 $ww^T+R$ 构造边缘高斯；回到第 03 章的网格积分方法。为使训练可以求导，使用 PyTorch，并在 log 空间计算。

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(42)
Y = torch.tensor(y, dtype=torch.float32)
sigma = .45
# 解码器输入一个标量 x，输出二维条件均值 f_theta(x)。
decoder = nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,2))
# 241 个隐变量取值，近似积分区间 [-6,6]。
grid = torch.linspace(-6,6,241).reshape(-1,1)



## 网格积分：单个点与整批计算

数学上先定义 $p(x)$、$p_\theta(y|x)$，再相乘得到 $p_\theta(x,y)$。对单个观测 y，枚举 x 网格即可积分。下面先给出逐点版本；训练版本仅将同一观测的多个网格点一起计算，以避免每次训练创建数万次小型计算图。两者公式相同。

二维条件分布的两个坐标在给定 x 后独立，因此 \(\log p(y|x)=\log p(y_1|x)+\log p(y_2|x)\)。代码用 Independent 明确表示两个坐标共同组成一次二维观测，而非两个样本。logsumexp 计算 \(\log\sum_j\exp(a_j)\)，避免极小密度的浮点下溢。

In [ ]:
def point_log_joint(x, y_i):
    prior = torch.distributions.Normal(0., 1.)
    mean = decoder(x.reshape(1, 1))[0]
    conditional = torch.distributions.MultivariateNormal(
        mean, sigma**2 * torch.eye(2, dtype=mean.dtype, device=mean.device)
    )
    log_prior = prior.log_prob(x)
    log_conditional = conditional.log_prob(y_i)
    return log_prior + log_conditional

def point_log_marginal(y_i, grid):
    log_joint_values = []
    for x in grid[:, 0]:
        log_joint_values.append(point_log_joint(x, y_i))
    dx = grid[1, 0] - grid[0, 0]
    return torch.logsumexp(torch.stack(log_joint_values), dim=0) + torch.log(dx)
# grid 的每一行是一个标量隐变量，形状 [G, 1]。
def log_joint(obs, grid):
    # 每个观测 y 对应 G 个候选 x；返回 [N, G]。
    prior = torch.distributions.Normal(0., 1.)
    log_prior = prior.log_prob(grid[:, 0])
    means = decoder(grid)
    # 每个候选 x 定义一个二维高斯条件分布。
    # Independent 将两个独立坐标的 log 密度相加，
    # 因此 log_prob 返回每个二维向量的联合 log 密度。
    conditional = torch.distributions.Independent(
        torch.distributions.Normal(means, sigma), 1
    )
    rows = []
    for y_i in obs:
        # 对同一个 y_i，在 G 个条件分布上求值。
        repeated_y = y_i.repeat(len(grid), 1)
        log_conditional = conditional.log_prob(repeated_y)
        log_joint_values = log_prior + log_conditional
        rows.append(log_joint_values)
    return torch.stack(rows)

def grid_log_marginal(obs, grid):
    log_joint_values = log_joint(obs, grid)
    dx = grid[1, 0] - grid[0, 0]
    values = []
    for row in log_joint_values:
        # p(y) ≈ sum_j p(x_j,y) * dx。
        # logsumexp 避免先 exp 后求和时的浮点下溢。
        log_marginal = torch.logsumexp(row, dim=0) + torch.log(dx)
        values.append(log_marginal)
    return torch.stack(values)

# 验证逐点与批量写法计算同一个边缘密度。
with torch.no_grad():
    small_grid = grid[::12]
    a = point_log_marginal(Y[0], small_grid)
    b = grid_log_marginal(Y[:1], small_grid)[0]
    torch.testing.assert_close(a, b)


## 2. 直接 MLE
每轮随机选取 128 条观测，估计训练集的平均对数似然；这不是隐变量采样。隐变量积分仍使用完整一维网格。Adam 最小化负对数似然。

In [ ]:
# 与第 02、03 章相同：直接最小化负观测对数似然。
opt = torch.optim.Adam(decoder.parameters(), lr=.01)
history = []
for _ in range(350):
    # 每轮随机取 128 个观测来估计平均训练目标。
    batch = Y[torch.randint(len(Y), (128,))]
    ll = grid_log_marginal(batch, grid).mean()
    opt.zero_grad()
    (-ll).backward()
    opt.step()
    history.append(ll.item())
plot_history(history, 'mean log likelihood (grid)')
# 可选后验验证：不参与梯度 MLE。
with torch.no_grad():
    lj = log_joint(Y[:1], grid)[0]
    log_z = grid_log_marginal(Y[:1], grid)[0]
    posterior = []
    for value in lj:
        posterior.append(torch.exp(value-log_z).item())
    plt.plot(grid[:,0], posterior)
    plt.title('Posterior density for first observation')
    plt.show()
# 沿用前两章的先验采样、条件采样。
with torch.no_grad():
    samples = []
    for i in range(len(y)):
        x = torch.randn(1)
        conditional = torch.distributions.Normal(decoder(x), sigma)
        samples.append(conditional.sample().numpy())
    compare(y, np.array(samples))

In [ ]:
# 改变网格密度，观察同一组参数下的边缘对数密度是否趋于稳定。
with torch.no_grad():
    for size in [61,121,241,481,961]:
        g = torch.linspace(-6,6,size).reshape(-1,1)
        values = grid_log_marginal(Y[:10], g)
        print(size, values.mean().item())
    # 保存训练好的生成模型，第 05 章直接复用。
    torch.save(decoder.state_dict(), 'decoder_grid.pt')

## 小结
每维 241 个网格点，二维就需要 241² 个点，十维需要 241¹⁰ 个点。直接 MLE 的数学目标没有改变，但数值积分可能不再可行。第 04.5 章进一步区分逐点密度求值和边缘积分，并研究先验采样能否替代网格积分。